# Schema Design and Hallucinated Arguments

### Setup

In [14]:
def get_api_key(identifier: str = "ANTHROPIC_API_KEY") -> str:
    """Get the Anthropic API key from Colab's userdata or a .env file."""
    try:
        from google.colab import userdata
        API_KEY = userdata.get(identifier)
    except:
        from dotenv import load_dotenv
        import os
        load_dotenv()
        API_KEY = os.getenv(identifier)
    
    return API_KEY

def print_models(client):
    """Print the list of models available in the given client."""
    for m in client.models.list():
        print(m.id, "-", m.display_name)


In [15]:
# Anthropic Connection test
from anthropic import Anthropic

ANTHROPIC_API_KEY = get_api_key()

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)

print_models(anthropic_client)

claude-fable-5-1 - Claude Fable 5.1
claude-opus-5 - Claude Opus 5
claude-sonnet-5 - Claude Sonnet 5
claude-fable-5 - Claude Fable 5
claude-opus-4-8 - Claude Opus 4.8
claude-opus-4-7 - Claude Opus 4.7
claude-sonnet-4-6 - Claude Sonnet 4.6
claude-opus-4-6 - Claude Opus 4.6
claude-opus-4-5-20251101 - Claude Opus 4.5
claude-haiku-4-5-20251001 - Claude Haiku 4.5
claude-sonnet-4-5-20250929 - Claude Sonnet 4.5


## Experiment

In [16]:
import json

CLAUDE_MODEL = "claude-haiku-4-5"

# Deliberately under-specified queries. Each leans on something the schema has
# to pin down: a vague threshold, a relative date, a category word, a currency,
# an ordering, or a judgment the tool cannot make.
PROMPTS = [
    "Show me big restaurant purchases from last quarter",
    "What did I spend on travel in euros in March?",
    "Find my 5 largest charges since the start of the year",
    "Any subscriptions over $50 I should cancel?",
]


def call_tool(prompt, tool, system=None):
    """Send one prompt with one tool available. Return the calls and any prose."""
    kwargs = {"system": system} if system else {}
    response = anthropic_client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=2048,
        tools=[tool],
        messages=[{"role": "user", "content": prompt}],
        **kwargs,
    )
    return {
        "prompt": prompt,
        # No tool_choice: whether the model calls the tool at all is a finding.
        "calls": [{"name": b.name, "input": b.input} for b in response.content if b.type == "tool_use"],
        "text": " ".join(b.text for b in response.content if b.type == "text").strip(),
        "stop_reason": response.stop_reason,
    }


def run_experiment(tool, label, system=None):
    """Run every prompt against one schema and print what the model sent."""
    print(f"===== {label} schema: {tool['name']} =====\n")
    results = []
    for prompt in PROMPTS:
        result = call_tool(prompt, tool, system=system)
        results.append(result)
        print(f"> {prompt}")
        if not result["calls"]:
            print(f"  no tool call (stop_reason={result['stop_reason']}): {result['text'][:200]}")
        for call in result["calls"]:
            print(f"  {call['name']} <- {json.dumps(call['input'], indent=2)}")
        if result["calls"] and result["text"]:
            print(f"  [also said] {result['text'][:200]}")
        print()
    return results


In [17]:
# Schema design, v1: deliberately loose.
SEARCH_TX_LOOSE = {
    "name": "search_transactions",
    "description": "Search the user's transactions.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "filters": {"type": "object"},
            "limit": {"type": "string"},
        },
        "required": ["query"],
    },
}


In [18]:
# Example execution, round 1: the loose schema.
loose_results = run_experiment(SEARCH_TX_LOOSE, "loose")


===== loose schema: search_transactions =====

> Show me big restaurant purchases from last quarter
  search_transactions <- {
  "query": "restaurant",
  "filters": {
    "amount_min": 50,
    "quarter": "last"
  }
}
  [also said] I'll search for large restaurant purchases from the last quarter.

> What did I spend on travel in euros in March?
  search_transactions <- {
  "query": "travel in March euros",
  "filters": {
    "category": "travel",
    "currency": "EUR",
    "month": "March"
  }
}
  [also said] I'll search your transactions for travel expenses in euros during March.

> Find my 5 largest charges since the start of the year
  search_transactions <- {
  "query": "charges since start of year",
  "limit": "5",
  "filters": {
    "type": "charge",
    "sort": "amount_desc"
  }
}

> Any subscriptions over $50 I should cancel?
  search_transactions <- {
  "query": "subscriptions",
  "filters": {
    "amount": {
      "$gt": 50
    }
  }
}



In [19]:
# Schema design, v2: the same tool, tightened along five axes.
from datetime import date


TODAY_NOTE = f"Today's date is {date.today().isoformat()}."

CATEGORIES = [
    "groceries", "dining", "travel", "transport",
    "utilities", "entertainment", "subscriptions", "other",
]

SEARCH_TX_TIGHT = {
    "name": "search_transactions",
    "description": (
        "Search the account holder's posted transactions. Returns matching "
        "transactions sorted as requested. Only searches posted transactions; "
        "pending transactions are not visible to this tool."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "merchant": {
                "type": "string",
                "description": "Merchant name substring, case-insensitive. Omit to match any merchant.",
            },
            "category": {
                "type": "string",
                "enum": CATEGORIES,
                "description": "Transaction category. Use 'other' only if none of the listed categories apply.",
            },
            "start_date": {
                "type": "string",
                "format": "date",
                "description": "Inclusive start of the date range as YYYY-MM-DD. Resolve relative phrases such as 'last quarter' to a concrete date before calling.",
            },
            "end_date": {
                "type": "string",
                "format": "date",
                "description": "Inclusive end of the date range as YYYY-MM-DD.",
            },
            "min_amount": {
                "type": "number",
                "minimum": 0,
                "description": "Minimum transaction amount in the account currency. Omit if the user did not give a threshold; do not invent one.",
            },
            "max_amount": {
                "type": "number",
                "minimum": 0,
                "description": "Maximum transaction amount in the account currency.",
            },
            "currency": {
                "type": "string",
                "enum": ["USD", "EUR", "GBP"],
                "description": "Currency the amounts are expressed in. Defaults to the account currency, USD.",
            },
            "sort_by": {
                "type": "string",
                "enum": ["date", "amount"],
                "description": "Field to sort results by. Defaults to date.",
            },
            "sort_order": {
                "type": "string",
                "enum": ["asc", "desc"],
                "description": "Sort direction. Defaults to desc.",
            },
            "limit": {
                "type": "integer",
                "minimum": 1,
                "maximum": 100,
                "default": 20,
                "description": "Maximum number of transactions to return.",
            },
        },
        "required": ["start_date", "end_date"],
        "additionalProperties": False,
    },
}


In [20]:
# Example execution, round 2: the tight schema, then a side-by-side comparison.
tight_results = run_experiment(SEARCH_TX_TIGHT, "tight", system=TODAY_NOTE)


def validate_args(tool, args):
    """Check one tool-call payload against a schema. Returns a list of problems.

    Only inspects top-level fields -- which is itself the point: the loose
    schema's `filters` is an open object, so whatever the model puts inside it
    is invisible here. A schema can only catch what it describes.
    """
    schema = tool["input_schema"]
    props = schema.get("properties", {})
    problems = []

    for field in schema.get("required", []):
        if field not in args:
            problems.append(f"missing required field: {field}")

    for key, value in args.items():
        spec = props.get(key)
        if spec is None:
            problems.append(f"field not in schema: {key}={value!r}")
            continue
        expected = spec["type"]
        matches = {
            "string": isinstance(value, str),
            "number": isinstance(value, (int, float)) and not isinstance(value, bool),
            "integer": isinstance(value, int) and not isinstance(value, bool),
            "boolean": isinstance(value, bool),
            "object": isinstance(value, dict),
        }[expected]
        if not matches:
            problems.append(f"{key}: expected {expected}, got {type(value).__name__} ({value!r})")
            continue
        if "enum" in spec and value not in spec["enum"]:
            problems.append(f"{key}={value!r} not in {spec['enum']}")
        if "minimum" in spec and value < spec["minimum"]:
            problems.append(f"{key}={value} below minimum {spec['minimum']}")
        if "maximum" in spec and value > spec["maximum"]:
            problems.append(f"{key}={value} above maximum {spec['maximum']}")

    return problems


for loose, tight in zip(loose_results, tight_results):
    print("=" * 72)
    print(loose["prompt"])
    for label, result, tool in (("LOOSE", loose, SEARCH_TX_LOOSE), ("TIGHT", tight, SEARCH_TX_TIGHT)):
        print(f"\n  [{label}]")
        if not result["calls"]:
            print("    no tool call made")
            continue
        args = result["calls"][0]["input"]
        print("    " + json.dumps(args))
        for problem in validate_args(tool, args) or ["schema-valid"]:
            print(f"    - {problem}")
    print()




===== tight schema: search_transactions =====

> Show me big restaurant purchases from last quarter
  search_transactions <- {
  "start_date": "2026-07-01",
  "end_date": "2026-09-30",
  "category": "dining",
  "sort_by": "amount",
  "sort_order": "desc"
}
  [also said] I'll search for your restaurant purchases from the last quarter (July 1 - September 30, 2026), focusing on larger amounts.

> What did I spend on travel in euros in March?
  search_transactions <- {
  "start_date": "2026-03-01",
  "end_date": "2026-03-31",
  "category": "travel",
  "currency": "EUR"
}
  [also said] I'll search for your travel transactions in March 2026 in euros.

> Find my 5 largest charges since the start of the year
  search_transactions <- {
  "start_date": "2026-01-01",
  "end_date": "2026-09-16",
  "limit": 5,
  "sort_by": "amount",
  "sort_order": "desc"
}

> Any subscriptions over $50 I should cancel?
  search_transactions <- {
  "start_date": "2026-01-01",
  "end_date": "2026-09-16",
  "category